# 05 - Statistical Analysis

This notebook performs statistical tests on e-commerce data:
- Kruskal-Wallis tests on AOV by channel and country
- Shapiro-Wilk normality tests
- Displays results from `outputs/metrics/statistical_tests.json`
- Displays correlation matrix PNG

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
from scipy import stats

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("Libraries loaded successfully.")

## 2. Load Data

In [ ]:
processed_path = Path('../data/processed')

orders = pd.read_csv(processed_path / 'orders_processed.csv')
orders["order_date"] = pd.to_datetime(orders["order_date"])

customers = pd.read_csv(processed_path / 'customers_processed.csv')

print(f'Orders: {orders.shape}')
print(f'Customers: {customers.shape}')

## 3. Descriptive Statistics

In [ ]:
print('Order Revenue Statistics:')
print(f"  Mean: ${orders['revenue'].mean():,.2f}")
print(f"  Median: ${orders['revenue'].median():,.2f}")
print(f"  Std: ${orders['revenue'].std():,.2f}")
print(f"  Min: ${orders['revenue'].min():,.2f}")
print(f"  Max: ${orders['revenue'].max():,.2f}")
print(f"  Skewness: {orders['revenue'].skew():.3f}")
print(f"  Kurtosis: {orders['revenue'].kurtosis():.3f}")

## 4. Shapiro-Wilk Normality Test

In [ ]:
sample_size = min(5000, len(orders))
revenue_sample = orders['revenue'].sample(n=sample_size, random_state=42)

shapiro_stat, shapiro_p = stats.shapiro(revenue_sample)

print("Shapiro-Wilk Normality Test on Order Revenue")
print("=" * 50)
print(f"Test statistic: {shapiro_stat:.4f}")
print(f"p-value: {shapiro_p:.6f}")
is_normal = shapiro_p > 0.05
print(f"\nConclusion: Revenue distribution is {'normal' if is_normal else 'not normal'} (alpha=0.05)")

## 5. Kruskal-Wallis Test by Channel

In [ ]:
if 'channel' in orders.columns:
    channels = orders['channel'].unique()
    channel_groups = [orders[orders['channel'] == ch]['revenue'].values for ch in channels]
    channel_groups = [g for g in channel_groups if len(g) > 0]
    
    if len(channel_groups) > 1:
        kw_stat_chan, kw_p_chan = stats.kruskal(*channel_groups)
        
        print("Kruskal-Wallis Test: AOV by Channel")
        print("=" * 50)
        print(f"Test statistic: {kw_stat_chan:.4f}")
        print(f"p-value: {kw_p_chan:.6f}")
        is_diff_chan = kw_p_chan < 0.05
        print(f"\nConclusion: Channel means are {'different' if is_diff_chan else 'not different'} (alpha=0.05)")
        
        channel_stats = orders.groupby('channel')['revenue'].agg(['count', 'mean', 'median', 'std'])
        channel_stats.columns = ['Count', 'Mean', 'Median', 'Std']
        print(f'\nChannel Statistics:')
        print(channel_stats.round(2).to_string())
    else:
        print('Not enough channel groups for Kruskal-Wallis test.')
else:
    print('No channel column found in orders data.')

## 6. Kruskal-Wallis Test by Country

In [ ]:
if 'country' in orders.columns:
    top_countries = orders['country'].value_counts().head(10).index.tolist()
    country_data = orders[orders['country'].isin(top_countries)]
    
    countries = country_data['country'].unique()
    country_groups = [country_data[country_data['country'] == c]['revenue'].values for c in countries]
    country_groups = [g for g in country_groups if len(g) > 0]
    
    if len(country_groups) > 1:
        kw_stat_ctry, kw_p_ctry = stats.kruskal(*country_groups)
        
        print("Kruskal-Wallis Test: AOV by Country (Top 10)")
        print("=" * 50)
        print(f"Test statistic: {kw_stat_ctry:.4f}")
        print(f"p-value: {kw_p_ctry:.6f}")
        is_diff_ctry = kw_p_ctry < 0.05
        print(f"\nConclusion: Country means are {'different' if is_diff_ctry else 'not different'} (alpha=0.05)")
        
        country_stats = country_data.groupby('country')['revenue'].agg(['count', 'mean', 'median', 'std'])
        country_stats.columns = ['Count', 'Mean', 'Median', 'Std']
        print(f'\nCountry Statistics (Top 10):')
        print(country_stats.round(2).to_string())
    else:
        print('Not enough country groups for Kruskal-Wallis test.')
else:
    print('No country column found in orders data.')

## 7. Statistical Tests Report

In [ ]:
metrics_path = Path('../outputs/metrics')

tests_path = metrics_path / 'statistical_tests.json'
if tests_path.exists():
    with open(tests_path, "r") as f:
        test_results = json.load(f)
    print("STATISTICAL TESTS RESULTS")
    print("=" * 60)
    print(json.dumps(test_results, indent=2, default=str))
else:
    print("statistical_tests.json not found.")

## 8. Display Correlation Matrix

In [ ]:
from IPython.display import Image, display

figures_path = Path('../outputs/figures')
corr_path = figures_path / 'correlation_matrix.png'

if corr_path.exists():
    print('Correlation Matrix from saved figure:')
    display(Image(filename=str(corr_path)))
else:
    numeric_cols = orders.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) > 1:
        corr_matrix = orders[numeric_cols].corr()
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
        ax.set_title('Correlation Matrix')
        plt.tight_layout()
        plt.show()

## 9. Summary

In [ ]:
print('Statistical Analysis Summary')
print("=" * 60)
print(f"Shapiro-Wilk p-value: {shapiro_p:.6f}")
is_normal = shapiro_p > 0.05
print(f"Revenue is {'normal' if is_normal else 'not normal'}")
if "channel" in orders.columns:
    print(f"Kruskal-Wallis (Channel) p-value: {kw_p_chan:.6f}")
if "country" in orders.columns:
    print(f"Kruskal-Wallis (Country) p-value: {kw_p_ctry:.6f}")
print(f"\nKey insight: Non-parametric tests are appropriate for this data.")